In [1]:
import numpy as np
import spacy
import warnings
import os
import time
import pandas as pd
from dotenv import load_dotenv
from scipy.optimize import minimize
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import logging

# --- Together AI Client ---
from together import Together

# Suppress warnings for clean terminal output
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# --- Qiskit Imports ---
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import Sampler as LocalSampler

# --- Configuration ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define the old CSV to read from and the new CSV to write to
OLD_CSV_FILENAME = "qrag_telemetry_N150_run_1783611471_final.csv"  # <-- UPDATE THIS TO YOUR PREVIOUS RUN'S CSV
RUN_TIMESTAMP = int(time.time())
CSV_FILENAME = f"qrag_telemetry_Updated_run_{RUN_TIMESTAMP}.csv"

# Load environment variables
load_dotenv()

# ==============================================================================
# DATASET PLACEHOLDER (NEW AGENT-PATIENT INVERSION SENTENCES)
# ==============================================================================
NEW_DATABASE = [
  {
    "class": "Agent-Patient Inversion",
    "text": "The crushed garlic squeezed the heavy garlic press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the crushed garlic",
    "conflict": "the heavy garlic press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed vegetables boiled the bamboo vegetable steamer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed vegetables",
    "conflict": "the bamboo vegetable steamer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cracked nuts snapped the heavy metal nutcracker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cracked nuts",
    "conflict": "the heavy metal nutcracker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed garment smoothed the handheld garment steamer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed garment",
    "conflict": "the handheld garment steamer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut glass scored the diamond-tipped glass cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut glass",
    "conflict": "the diamond-tipped glass cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted solder wicked the copper desoldering braid.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted solder",
    "conflict": "the copper desoldering braid"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut tile snapped the manual tile cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut tile",
    "conflict": "the manual tile cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The poured driveway floated the magnesium bull float.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the poured driveway",
    "conflict": "the magnesium bull float"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The secured drywall screwed the electric drywall gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the secured drywall",
    "conflict": "the electric drywall gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut cardboard scissored the sharp craft scissors.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut cardboard",
    "conflict": "the sharp craft scissors"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The incubated cells warmed the regulated cell incubator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the incubated cells",
    "conflict": "the regulated cell incubator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The trimmed hedge sheared the electric hedge trimmer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the trimmed hedge",
    "conflict": "the electric hedge trimmer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled champagne cooled the thermoelectric wine cooler fridge.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled champagne",
    "conflict": "the thermoelectric wine cooler fridge"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried towels spun the electric vented drum clothes dryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried towels",
    "conflict": "the electric vented drum clothes dryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The circulated breeze blew the rotating oscillating pedestal standing fan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the circulated breeze",
    "conflict": "the rotating oscillating pedestal standing fan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The whipped cream folded the silicone mixing spatula.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the whipped cream",
    "conflict": "the silicone mixing spatula"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted butter dissolved the stainless steel saucepan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted butter",
    "conflict": "the stainless steel saucepan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The roasted beans ground the ceramic coffee burr.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the roasted beans",
    "conflict": "the ceramic coffee burr"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baked bread warmed the heavy stone baking peel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baked bread",
    "conflict": "the heavy stone baking peel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chilled dough rested the wooden proofing basket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chilled dough",
    "conflict": "the wooden proofing basket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The fermented cabbage soured the ceramic pickling crock.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the fermented cabbage",
    "conflict": "the ceramic pickling crock"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The frozen ice cracked the plastic ice cube tray.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the frozen ice",
    "conflict": "the plastic ice cube tray"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pureed soup blended the immersion hand blender.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pureed soup",
    "conflict": "the immersion hand blender"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caramelized sugar burned the copper candy pot.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caramelized sugar",
    "conflict": "the copper candy pot"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The fermented wine aged the charred oak barrel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the fermented wine",
    "conflict": "the charred oak barrel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The brewed tea steeped the mesh infuser ball.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the brewed tea",
    "conflict": "the mesh infuser ball"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charred steak seared the cast iron grill grate.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charred steak",
    "conflict": "the cast iron grill grate"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The toasted nuts roasted the flat aluminum baking sheet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the toasted nuts",
    "conflict": "the flat aluminum baking sheet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried fruit dehydrated the tiered food dehydrator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried fruit",
    "conflict": "the tiered food dehydrator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The kneaded dough stretched the automatic stand mixer dough hook.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the kneaded dough",
    "conflict": "the automatic stand mixer dough hook"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The filtered water dripped the carbon water filter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the filtered water",
    "conflict": "the carbon water filter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed batter blended the rotary hand mixer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed batter",
    "conflict": "the rotary hand mixer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The separated curds split the fine cheesecloth.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the separated curds",
    "conflict": "the fine cheesecloth"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dissolved salt melted the wooden stirring spoon.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dissolved salt",
    "conflict": "the wooden stirring spoon"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The whipped egg whites stiffened the copper mixing bowl.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the whipped egg whites",
    "conflict": "the copper mixing bowl"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped garlic scattered the wooden cutting board.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped garlic",
    "conflict": "the wooden cutting board"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sliced ham cured the heavy butcher block.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sliced ham",
    "conflict": "the heavy butcher block"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The minced herbs gathered the steel mezzaluna knife.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the minced herbs",
    "conflict": "the steel mezzaluna knife"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The peeled potato stripped the vertical vegetable peeler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the peeled potato",
    "conflict": "the vertical vegetable peeler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cored apple split the tubular apple corer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cored apple",
    "conflict": "the tubular apple corer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The squeezed orange burst the manual citrus press.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the squeezed orange",
    "conflict": "the manual citrus press"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ground meat turned the heavy manual meat grinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ground meat",
    "conflict": "the heavy manual meat grinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The whisked matcha frothed the traditional bamboo whisk.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the whisked matcha",
    "conflict": "the traditional bamboo whisk"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sifted flour dusted the rotating flour sifter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sifted flour",
    "conflict": "the rotating flour sifter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The strained pasta drained the steel metal colander.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the strained pasta",
    "conflict": "the steel metal colander"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rolled pastry flattened the marble pastry roller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rolled pastry",
    "conflict": "the marble pastry roller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured flour filled the plastic measuring cup.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured flour",
    "conflict": "the plastic measuring cup"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The portioned rice scooped the wooden rice paddle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the portioned rice",
    "conflict": "the wooden rice paddle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mashed potato squashed the heavy wire potato masher.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mashed potato",
    "conflict": "the heavy wire potato masher"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The flipped pancake turned the flexible nylon turner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the flipped pancake",
    "conflict": "the flexible nylon turner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sanded wood smoothed the random orbital sander.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sanded wood",
    "conflict": "the random orbital sander"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sawed plank split the electric circular saw.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sawed plank",
    "conflict": "the electric circular saw"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The nailed board fastened the pneumatic brad nailer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the nailed board",
    "conflict": "the pneumatic brad nailer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The screwed joint tightened the cordless impact driver.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the screwed joint",
    "conflict": "the cordless impact driver"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The glued joint bound the steel bar clamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the glued joint",
    "conflict": "the steel bar clamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chiseled mortise split the bevel-edge wood chisel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chiseled mortise",
    "conflict": "the bevel-edge wood chisel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The planed board shaved the hand block plane.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the planed board",
    "conflict": "the hand block plane"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The routed groove cut the variable speed plunge router.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the routed groove",
    "conflict": "the variable speed plunge router"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drilled hole bored the titanium coated drill bit.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drilled hole",
    "conflict": "the titanium coated drill bit"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured gap spanned the steel vernier caliper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured gap",
    "conflict": "the steel vernier caliper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The leveled shelf balanced the laser level tool.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the leveled shelf",
    "conflict": "the laser level tool"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The welded seam fused the MIG welding gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the welded seam",
    "conflict": "the MIG welding gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The soldered pipe melted the propane soldering torch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the soldered pipe",
    "conflict": "the propane soldering torch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bolted flange tightened the adjustable crescent wrench.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bolted flange",
    "conflict": "the adjustable crescent wrench"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The riveted sheet popped the heavy manual pop riveter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the riveted sheet",
    "conflict": "the heavy manual pop riveter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The painted trim coated the synthetic angled paint brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the painted trim",
    "conflict": "the synthetic angled paint brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stained deck darkened the wool stain applicator pad.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stained deck",
    "conflict": "the wool stain applicator pad"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stripped paint peeled the high carbon paint scraper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stripped paint",
    "conflict": "the high carbon paint scraper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The plastered wall smoothed the flat steel finishing trowel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the plastered wall",
    "conflict": "the flat steel finishing trowel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caulked seam filled the dripless manual caulk gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caulked seam",
    "conflict": "the dripless manual caulk gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut pipe snapped the ratcheting PVC pipe cutter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut pipe",
    "conflict": "the ratcheting PVC pipe cutter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bent conduit curved the manual steel conduit bender.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bent conduit",
    "conflict": "the manual steel conduit bender"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stripped wire exposed the automatic wire stripper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stripped wire",
    "conflict": "the automatic wire stripper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The crimped connector squeezed the ratcheting wire crimper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the crimped connector",
    "conflict": "the ratcheting wire crimper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stapled wire fastened the heavy duty staple gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stapled wire",
    "conflict": "the heavy duty staple gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The polished brass shined the rotary buffing wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the polished brass",
    "conflict": "the rotary buffing wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sharpened blade ground the slow speed bench grinder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sharpened blade",
    "conflict": "the slow speed bench grinder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The engraved metal scratched the pneumatic engraving pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the engraved metal",
    "conflict": "the pneumatic engraving pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The carved wood shaped the high speed rotary tool.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the carved wood",
    "conflict": "the high speed rotary tool"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The forged iron bent the heavy blacksmith anvil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the forged iron",
    "conflict": "the heavy blacksmith anvil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The melted bronze poured the graphite melting crucible.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the melted bronze",
    "conflict": "the graphite melting crucible"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mixed concrete cured the electric portable cement mixer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mixed concrete",
    "conflict": "the electric portable cement mixer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The compacted soil settled the heavy steel tamper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the compacted soil",
    "conflict": "the heavy steel tamper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dug posthole opened the dual handled post hole digger.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dug posthole",
    "conflict": "the dual handled post hole digger"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The swept floor cleared the wide industrial push broom.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the swept floor",
    "conflict": "the wide industrial push broom"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The vacuumed dust emptied the heavy wet dry vacuum.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the vacuumed dust",
    "conflict": "the heavy wet dry vacuum"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed window cleared the rubber window squeegee.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed window",
    "conflict": "the rubber window squeegee"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mopped spill dried the industrial cotton loop mop.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mopped spill",
    "conflict": "the industrial cotton loop mop"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The secured load bound the ratcheting cargo tie down strap.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the secured load",
    "conflict": "the ratcheting cargo tie down strap"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hoisted engine lifted the hydraulic steel engine crane.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hoisted engine",
    "conflict": "the hydraulic steel engine crane"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The jacked car raised the hydraulic low profile floor jack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the jacked car",
    "conflict": "the hydraulic low profile floor jack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The inflated tire expanded the portable electric air compressor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the inflated tire",
    "conflict": "the portable electric air compressor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The lubricated chain oiled the precision needle oiler bottle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the lubricated chain",
    "conflict": "the precision needle oiler bottle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The greased bearing filled the heavy pneumatic grease gun.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the greased bearing",
    "conflict": "the heavy pneumatic grease gun"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The jumped battery sparked the insulated copper jumper cables.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the jumped battery",
    "conflict": "the insulated copper jumper cables"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charged phone powered the braided USB charging cable.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charged phone",
    "conflict": "the braided USB charging cable"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated workspace brightened the halogen dual head work light.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated workspace",
    "conflict": "the halogen dual head work light"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ventilated fumes blew the high velocity industrial floor fan.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ventilated fumes",
    "conflict": "the high velocity industrial floor fan"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The heated shop warmed the propane forced air heater.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the heated shop",
    "conflict": "the propane forced air heater"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooled room chilled the portable evaporative swamp cooler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooled room",
    "conflict": "the portable evaporative swamp cooler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The printed page inked the wireless laser office printer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the printed page",
    "conflict": "the wireless laser office printer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scanned document digitized the flatbed optical document scanner.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scanned document",
    "conflict": "the flatbed optical document scanner"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shredded paper tore the micro cut document shredder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shredded paper",
    "conflict": "the micro cut document shredder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The laminated poster sealed the thermal pouch laminator machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the laminated poster",
    "conflict": "the thermal pouch laminator machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bound book folded the heavy duty wire binder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bound book",
    "conflict": "the heavy duty wire binder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stapled packet pierced the ergonomic desktop stapler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stapled packet",
    "conflict": "the ergonomic desktop stapler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The punched paper popped the manual three hole punch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the punched paper",
    "conflict": "the manual three hole punch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clipped paper gathered the giant steel binder clip.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clipped paper",
    "conflict": "the giant steel binder clip"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pinned note stuck the sharp brass push pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pinned note",
    "conflict": "the sharp brass push pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The taped box sealed the handheld packing tape dispenser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the taped box",
    "conflict": "the handheld packing tape dispenser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The glued envelope stuck the non-toxic washable glue stick.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the glued envelope",
    "conflict": "the non-toxic washable glue stick"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The erased mistake rubbed the white vinyl block eraser.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the erased mistake",
    "conflict": "the white vinyl block eraser"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The highlighted text glowed the fluorescent chisel tip highlighter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the highlighted text",
    "conflict": "the fluorescent chisel tip highlighter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The written letter flowed the gold nibbed fountain pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the written letter",
    "conflict": "the gold nibbed fountain pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drafted plan drew the thin lead mechanical pencil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drafted plan",
    "conflict": "the thin lead mechanical pencil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The marked cardboard darkened the thick felt tip permanent marker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the marked cardboard",
    "conflict": "the thick felt tip permanent marker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured drawing scaled the triangular aluminum architect scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured drawing",
    "conflict": "the triangular aluminum architect scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut photo snapped the heavy rotary paper trimmer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut photo",
    "conflict": "the heavy rotary paper trimmer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The organized files sorted the expanding accordion file folder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the organized files",
    "conflict": "the expanding accordion file folder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The labeled folder printed the handheld thermal label maker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the labeled folder",
    "conflict": "the handheld thermal label maker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stamped date imprinted the automatic self inking date stamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stamped date",
    "conflict": "the automatic self inking date stamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mailed letter sealed the sticky self sealing envelope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mailed letter",
    "conflict": "the sticky self sealing envelope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weighed package balanced the digital postage shipping scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weighed package",
    "conflict": "the digital postage shipping scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The calculated sum totaled the solar powered desktop calculator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the calculated sum",
    "conflict": "the solar powered desktop calculator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The typed report clicked the mechanical switch computer keyboard.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the typed report",
    "conflict": "the mechanical switch computer keyboard"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clicked link tracked the wireless optical computer mouse.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clicked link",
    "conflict": "the wireless optical computer mouse"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tapped screen registered the active digital smart stylus.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tapped screen",
    "conflict": "the active digital smart stylus"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The projected image beamed the high lumen presentation projector.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the projected image",
    "conflict": "the high lumen presentation projector"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The recorded audio captured the omnidirectional studio condenser microphone.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the recorded audio",
    "conflict": "the omnidirectional studio condenser microphone"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The photographed scene flashed the digital mirrorless camera body.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the photographed scene",
    "conflict": "the digital mirrorless camera body"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The filmed video rolled the stabilized handheld action camera.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the filmed video",
    "conflict": "the stabilized handheld action camera"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated subject brightened the circular LED ring light.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated subject",
    "conflict": "the circular LED ring light"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charged laptop powered the bulky AC power adapter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charged laptop",
    "conflict": "the bulky AC power adapter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stored data saved the portable external solid state drive.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stored data",
    "conflict": "the portable external solid state drive"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The networked server connected the high speed gigabit ethernet router.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the networked server",
    "conflict": "the high speed gigabit ethernet router"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooled processor chilled the liquid CPU cooling radiator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooled processor",
    "conflict": "the liquid CPU cooling radiator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The displayed graphic glowed the high resolution IPS monitor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the displayed graphic",
    "conflict": "the high resolution IPS monitor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The broadcasted signal transmitted the outdoor directional Wi-Fi antenna.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the broadcasted signal",
    "conflict": "the outdoor directional Wi-Fi antenna"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shredded disc broke the heavy duty optical media shredder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shredded disc",
    "conflict": "the heavy duty optical media shredder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The read book opened the leather bound hardcover novel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the read book",
    "conflict": "the leather bound hardcover novel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bookmarked page flipped the woven silk ribbon bookmark.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bookmarked page",
    "conflict": "the woven silk ribbon bookmark"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened letter tore the brass sword letter opener.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened letter",
    "conflict": "the brass sword letter opener"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sorted mail filled the tiered metal desktop file organizer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sorted mail",
    "conflict": "the tiered metal desktop file organizer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The presented chart folded the collapsible aluminum presentation easel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the presented chart",
    "conflict": "the collapsible aluminum presentation easel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shredded manuscript destroyed the heavy cross cut paper shredder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shredded manuscript",
    "conflict": "the heavy cross cut paper shredder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The backed-up file synced the remote cloud storage server.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the backed-up file",
    "conflict": "the remote cloud storage server"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The formatted drive erased the portable external hard drive.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the formatted drive",
    "conflict": "the portable external hard drive"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The signed document validated the electronic digital signature pad.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the signed document",
    "conflict": "the electronic digital signature pad"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pitched presentation showed the digital smart interactive whiteboard.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pitched presentation",
    "conflict": "the digital smart interactive whiteboard"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The transmitted fax sent the analog dial up fax machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the transmitted fax",
    "conflict": "the analog dial up fax machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The watered grass soaked the oscillating lawn sprinkler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the watered grass",
    "conflict": "the oscillating lawn sprinkler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The mowed lawn clipped the gas powered rotary push mower.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the mowed lawn",
    "conflict": "the gas powered rotary push mower"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The trimmed edge snipped the electric string weed trimmer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the trimmed edge",
    "conflict": "the electric string weed trimmer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pruned branch snapped the long handled bypass loppers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pruned branch",
    "conflict": "the long handled bypass loppers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sawed limb broke the folding steel pruning saw.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sawed limb",
    "conflict": "the folding steel pruning saw"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped wood split the heavy forged felling axe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped wood",
    "conflict": "the heavy forged felling axe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dug hole opened the long handled pointed shovel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dug hole",
    "conflict": "the long handled pointed shovel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The turned soil loosened the four tine steel pitchfork.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the turned soil",
    "conflict": "the four tine steel pitchfork"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The raked leaves gathered the flexible plastic leaf rake.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the raked leaves",
    "conflict": "the flexible plastic leaf rake"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The swept driveway cleared the stiff bristle outdoor push broom.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the swept driveway",
    "conflict": "the stiff bristle outdoor push broom"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The blown debris scattered the gas powered backpack leaf blower.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the blown debris",
    "conflict": "the gas powered backpack leaf blower"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed patio sprayed the high pressure gas power washer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed patio",
    "conflict": "the high pressure gas power washer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The planted seed grew the plastic seedling starter tray.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the planted seed",
    "conflict": "the plastic seedling starter tray"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The fertilized bed bloomed the wheeled broadcast fertilizer spreader.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the fertilized bed",
    "conflict": "the wheeled broadcast fertilizer spreader"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The composted waste rotted the tumbling plastic compost bin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the composted waste",
    "conflict": "the tumbling plastic compost bin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The harvested apple fell the wire fruit picking basket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the harvested apple",
    "conflict": "the wire fruit picking basket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weeded garden cleared the long handled oscillating stirrup hoe.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weeded garden",
    "conflict": "the long handled oscillating stirrup hoe"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tilled earth turned the powerful gas powered garden tiller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tilled earth",
    "conflict": "the powerful gas powered garden tiller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The edged sidewalk cut the heavy steel half moon edger.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the edged sidewalk",
    "conflict": "the heavy steel half moon edger"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hauled dirt rolled the pneumatic tire steel wheelbarrow.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hauled dirt",
    "conflict": "the pneumatic tire steel wheelbarrow"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped brush shattered the heavy steel brush machete.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped brush",
    "conflict": "the heavy steel brush machete"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The split log cracked the hydraulic motorized log splitter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the split log",
    "conflict": "the hydraulic motorized log splitter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chipped branch shredded the powerful gas wood chipper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chipped branch",
    "conflict": "the powerful gas wood chipper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The caught fish splashed the lightweight carbon fiber fishing rod.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the caught fish",
    "conflict": "the lightweight carbon fiber fishing rod"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The reeled line snapped the aluminum spinning fishing reel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the reeled line",
    "conflict": "the aluminum spinning fishing reel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The netted catch trapped the rubberized landing fish net.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the netted catch",
    "conflict": "the rubberized landing fish net"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The baited hook sank the sharp barbed circle fish hook.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the baited hook",
    "conflict": "the sharp barbed circle fish hook"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pitched tent popped the flexible fiberglass tent pole.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pitched tent",
    "conflict": "the flexible fiberglass tent pole"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The staked guyline tightened the reflective nylon tent cord.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the staked guyline",
    "conflict": "the reflective nylon tent cord"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The lit fire burned the windproof butane camping lighter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the lit fire",
    "conflict": "the windproof butane camping lighter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooked meal boiled the portable propane camping stove.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooked meal",
    "conflict": "the portable propane camping stove"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The purified water filtered the ceramic pump backpacking water filter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the purified water",
    "conflict": "the ceramic pump backpacking water filter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated path beamed the rechargeable LED headlamp.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated path",
    "conflict": "the rechargeable LED headlamp"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The chopped kindling shattered the compact camping survival hatchet.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the chopped kindling",
    "conflict": "the compact camping survival hatchet"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The navigated trail pointed the liquid filled magnetic orienting compass.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the navigated trail",
    "conflict": "the liquid filled magnetic orienting compass"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The tracked steps counted the digital wearable fitness tracker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the tracked steps",
    "conflict": "the digital wearable fitness tracker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scoped deer magnified the variable zoom hunting riflescope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scoped deer",
    "conflict": "the variable zoom hunting riflescope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shot target pierced the compound mechanical hunting bow.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shot target",
    "conflict": "the compound mechanical hunting bow"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The trapped pest snapped the wooden spring loaded mousetrap.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the trapped pest",
    "conflict": "the wooden spring loaded mousetrap"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The repelled mosquito smoked the slow burning citronella coil.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the repelled mosquito",
    "conflict": "the slow burning citronella coil"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The fed bird scattered the hanging plastic tube bird feeder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the fed bird",
    "conflict": "the hanging plastic tube bird feeder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The watered flower dripped the slow release terracotta watering spike.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the watered flower",
    "conflict": "the slow release terracotta watering spike"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported tomato wrapped the galvanized steel round tomato cage.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported tomato",
    "conflict": "the galvanized steel round tomato cage"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shaded deck blocked the retractable canvas patio awning.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shaded deck",
    "conflict": "the retractable canvas patio awning"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The heated patio warmed the tall propane outdoor mushroom heater.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the heated patio",
    "conflict": "the tall propane outdoor mushroom heater"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The smoked meat charred the heavy offset barrel wood smoker.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the smoked meat",
    "conflict": "the heavy offset barrel wood smoker"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The grilled burger sizzled the cast iron tabletop charcoal grill.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the grilled burger",
    "conflict": "the cast iron tabletop charcoal grill"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The flipped steak turned the long handled stainless steel grilling tongs.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the flipped steak",
    "conflict": "the long handled stainless steel grilling tongs"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The basted rib dripped the silicone bristle barbecue basting brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the basted rib",
    "conflict": "the silicone bristle barbecue basting brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The probed roast beeped the digital instant read meat thermometer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the probed roast",
    "conflict": "the digital instant read meat thermometer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steered car turned the leather wrapped steering wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steered car",
    "conflict": "the leather wrapped steering wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braked truck stopped the hydraulic disc brake caliper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braked truck",
    "conflict": "the hydraulic disc brake caliper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The accelerated coupe roared the electronic fuel injection throttle body.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the accelerated coupe",
    "conflict": "the electronic fuel injection throttle body"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shifted gear changed the dual clutch automatic transmission.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shifted gear",
    "conflict": "the dual clutch automatic transmission"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The wiped glass cleared the silicone beam windshield wiper.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the wiped glass",
    "conflict": "the silicone beam windshield wiper"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed car shone the foaming automotive wash mitt.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed car",
    "conflict": "the foaming automotive wash mitt"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The waxed paint gleamed the dual action random orbital polisher.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the waxed paint",
    "conflict": "the dual action random orbital polisher"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The towed trailer pulled the heavy duty steel trailer hitch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the towed trailer",
    "conflict": "the heavy duty steel trailer hitch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The secured cargo strapped the heavy woven nylon ratchet strap.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the secured cargo",
    "conflict": "the heavy woven nylon ratchet strap"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The navigated route mapped the dashboard mounted satellite GPS.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the navigated route",
    "conflict": "the dashboard mounted satellite GPS"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The honked horn blared the steering wheel center horn pad.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the honked horn",
    "conflict": "the steering wheel center horn pad"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The fueled tank filled the unleaded gasoline pump nozzle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the fueled tank",
    "conflict": "the unleaded gasoline pump nozzle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The charged battery powered the fast direct current electric charger.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the charged battery",
    "conflict": "the fast direct current electric charger"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The locked door clicked the remote keyless entry fob.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the locked door",
    "conflict": "the remote keyless entry fob"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The opened window dropped the electric power window switch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the opened window",
    "conflict": "the electric power window switch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The adjusted mirror turned the motorized side view mirror switch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the adjusted mirror",
    "conflict": "the motorized side view mirror switch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The heated seat warmed the integrated wire seat heating element.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the heated seat",
    "conflict": "the integrated wire seat heating element"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cooled cabin chilled the automotive air conditioning compressor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cooled cabin",
    "conflict": "the automotive air conditioning compressor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The exhausted smoke vented the stainless steel performance muffler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the exhausted smoke",
    "conflict": "the stainless steel performance muffler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated road brightened the high intensity discharge headlight.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated road",
    "conflict": "the high intensity discharge headlight"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pedaled bike turned the forged aluminum bicycle crankset.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pedaled bike",
    "conflict": "the forged aluminum bicycle crankset"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shifted chain moved the rear bicycle derailleur mechanism.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shifted chain",
    "conflict": "the rear bicycle derailleur mechanism"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braked wheel stopped the hydraulic bicycle disc brake.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braked wheel",
    "conflict": "the hydraulic bicycle disc brake"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steered wheel turned the carbon fiber bicycle handlebar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steered wheel",
    "conflict": "the carbon fiber bicycle handlebar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The inflated tube expanded the high pressure bicycle floor pump.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the inflated tube",
    "conflict": "the high pressure bicycle floor pump"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The patched tire sealed the vulcanizing rubber tire patch kit.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the patched tire",
    "conflict": "the vulcanizing rubber tire patch kit"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The locked frame secured the heavy hardened steel U-lock.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the locked frame",
    "conflict": "the heavy hardened steel U-lock"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated path flashed the rechargeable LED bicycle headlight.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated path",
    "conflict": "the rechargeable LED bicycle headlight"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The rowed boat paddled the long wooden rowboat oar.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the rowed boat",
    "conflict": "the long wooden rowboat oar"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sailed ship caught the heavy canvas mainsail.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sailed ship",
    "conflict": "the heavy canvas mainsail"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steered yacht turned the wooden ship steering wheel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steered yacht",
    "conflict": "the wooden ship steering wheel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The anchored vessel dropped the heavy galvanized steel plow anchor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the anchored vessel",
    "conflict": "the heavy galvanized steel plow anchor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The moored boat tied the heavy braided nylon dock line.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the moored boat",
    "conflict": "the heavy braided nylon dock line"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drained bilge pumped the submersible automatic bilge pump.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drained bilge",
    "conflict": "the submersible automatic bilge pump"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The navigated sea charted the dashboard marine chartplotter.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the navigated sea",
    "conflict": "the dashboard marine chartplotter"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sounded depth beeped the acoustic marine depth sounder.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sounded depth",
    "conflict": "the acoustic marine depth sounder"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The flown plane soared the aluminum aircraft wing flap.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the flown plane",
    "conflict": "the aluminum aircraft wing flap"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steered jet turned the pilot control flight yoke.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steered jet",
    "conflict": "the pilot control flight yoke"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braked aircraft stopped the heavy aircraft landing gear.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braked aircraft",
    "conflict": "the heavy aircraft landing gear"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The accelerated jet roared the twin spool turbofan engine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the accelerated jet",
    "conflict": "the twin spool turbofan engine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The navigated flight plotted the integrated glass cockpit avionics.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the navigated flight",
    "conflict": "the integrated glass cockpit avionics"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hoisted cargo lifted the heavy duty industrial gantry crane.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hoisted cargo",
    "conflict": "the heavy duty industrial gantry crane"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The moved pallet rolled the manual hydraulic pallet jack.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the moved pallet",
    "conflict": "the manual hydraulic pallet jack"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The transported load drove the heavy diesel transport forklift.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the transported load",
    "conflict": "the heavy diesel transport forklift"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dumped soil tipped the hydraulic dump truck bed.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dumped soil",
    "conflict": "the hydraulic dump truck bed"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dug trench scraped the hydraulic crawler excavator bucket.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dug trench",
    "conflict": "the hydraulic crawler excavator bucket"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The graded road leveled the heavy diesel motor grader blade.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the graded road",
    "conflict": "the heavy diesel motor grader blade"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The paved asphalt rolled the heavy vibratory steel drum roller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the paved asphalt",
    "conflict": "the heavy vibratory steel drum roller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The plowed snow pushed the angled steel snow plow blade.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the plowed snow",
    "conflict": "the angled steel snow plow blade"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sanded ice scattered the automated truck mounted salt spreader.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sanded ice",
    "conflict": "the automated truck mounted salt spreader"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The injected medicine pierced the sterile hypodermic syringe needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the injected medicine",
    "conflict": "the sterile hypodermic syringe needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The drawn blood filled the vacuum blood collection tube.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the drawn blood",
    "conflict": "the vacuum blood collection tube"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The stitched wound closed the curved surgical suture needle.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the stitched wound",
    "conflict": "the curved surgical suture needle"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The bandaged cut wrapped the sterile self adhering bandage.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the bandaged cut",
    "conflict": "the sterile self adhering bandage"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The checked heartbeat thumped the acoustic cardiology stethoscope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the checked heartbeat",
    "conflict": "the acoustic cardiology stethoscope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The measured fever beeped the digital infrared ear thermometer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the measured fever",
    "conflict": "the digital infrared ear thermometer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The weighed patient balanced the digital medical floor scale.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the weighed patient",
    "conflict": "the digital medical floor scale"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The spun sample separated the high speed laboratory centrifuge.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the spun sample",
    "conflict": "the high speed laboratory centrifuge"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The transferred liquid dropped the adjustable volume mechanical pipettor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the transferred liquid",
    "conflict": "the adjustable volume mechanical pipettor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The magnified cell focused the binocular compound light microscope.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the magnified cell",
    "conflict": "the binocular compound light microscope"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The scanned bone radiated the digital medical x-ray machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the scanned bone",
    "conflict": "the digital medical x-ray machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The monitored heart beeped the electrical electrocardiogram machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the monitored heart",
    "conflict": "the electrical electrocardiogram machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The oxygenated blood pumped the external membrane oxygenation machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the oxygenated blood",
    "conflict": "the external membrane oxygenation machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ventilated lung breathed the mechanical hospital intensive care ventilator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ventilated lung",
    "conflict": "the mechanical hospital intensive care ventilator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shocked heart jumped the automated external automated defibrillator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shocked heart",
    "conflict": "the automated external automated defibrillator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The clamped artery stopped the locking surgical hemostat forceps.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the clamped artery",
    "conflict": "the locking surgical hemostat forceps"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cauterized tissue burned the handheld surgical electrocautery pen.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cauterized tissue",
    "conflict": "the handheld surgical electrocautery pen"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut skin split the sharp surgical carbon steel scalpel.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut skin",
    "conflict": "the sharp surgical carbon steel scalpel"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The retracted skin opened the locking steel surgical retractor.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the retracted skin",
    "conflict": "the locking steel surgical retractor"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The illuminated tissue glowed the overhead surgical theater light.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the illuminated tissue",
    "conflict": "the overhead surgical theater light"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The braced knee bent the hinged neoprene orthopedic knee brace.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the braced knee",
    "conflict": "the hinged neoprene orthopedic knee brace"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The casted arm hardened the synthetic fiberglass orthopedic cast.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the casted arm",
    "conflict": "the synthetic fiberglass orthopedic cast"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The supported step walked the adjustable aluminum medical crutch.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the supported step",
    "conflict": "the adjustable aluminum medical crutch"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pushed patient rolled the folding manual transit wheelchair.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pushed patient",
    "conflict": "the folding manual transit wheelchair"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cleared airway sucked the portable medical suction aspirator.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cleared airway",
    "conflict": "the portable medical suction aspirator"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed hands lathered the antibacterial foaming liquid hand soap.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed hands",
    "conflict": "the antibacterial foaming liquid hand soap"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried hands blew the high speed electric hand dryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried hands",
    "conflict": "the high speed electric hand dryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The brushed teeth cleaned the oscillating electric toothbrush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the brushed teeth",
    "conflict": "the oscillating electric toothbrush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The flossed gap snapped the mint flavored dental floss.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the flossed gap",
    "conflict": "the mint flavored dental floss"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The shaved beard clipped the rechargeable electric foil shaver.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the shaved beard",
    "conflict": "the rechargeable electric foil shaver"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The trimmed hair snipped the professional barber hair clippers.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the trimmed hair",
    "conflict": "the professional barber hair clippers"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried hair blew the hot ionic ceramic hair dryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried hair",
    "conflict": "the hot ionic ceramic hair dryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The straightened hair flattened the heated ceramic flat iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the straightened hair",
    "conflict": "the heated ceramic flat iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The curled hair wrapped the heated ceramic curling wand.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the curled hair",
    "conflict": "the heated ceramic curling wand"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The painted nail colored the glossy synthetic nail polish brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the painted nail",
    "conflict": "the glossy synthetic nail polish brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The filed nail smoothed the double sided emery board nail file.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the filed nail",
    "conflict": "the double sided emery board nail file"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed body scrubbed the exfoliating synthetic mesh bath sponge.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed body",
    "conflict": "the exfoliating synthetic mesh bath sponge"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The moisturized skin absorbed the hydrating daily body lotion.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the moisturized skin",
    "conflict": "the hydrating daily body lotion"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The applied makeup brushed the soft synthetic powder brush.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the applied makeup",
    "conflict": "the soft synthetic powder brush"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The curled lash bent the stainless steel mechanical eyelash curler.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the curled lash",
    "conflict": "the stainless steel mechanical eyelash curler"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The ironed shirt pressed the hot steam clothes iron.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the ironed shirt",
    "conflict": "the hot steam clothes iron"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The washed clothes tumbled the front load washing machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the washed clothes",
    "conflict": "the front load washing machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The dried clothes spun the vented electric clothes dryer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the dried clothes",
    "conflict": "the vented electric clothes dryer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The steamed dress smoothed the upright fabric garment steamer.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the steamed dress",
    "conflict": "the upright fabric garment steamer"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The folded shirt stacked the plastic laundry folding board.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the folded shirt",
    "conflict": "the plastic laundry folding board"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The hung jacket draped the wooden curved suit hanger.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the hung jacket",
    "conflict": "the wooden curved suit hanger"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The linted coat rolled the sticky adhesive lint roller.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the linted coat",
    "conflict": "the sticky adhesive lint roller"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The sewn fabric stitched the motorized electric sewing machine.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the sewn fabric",
    "conflict": "the motorized electric sewing machine"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The pinned hem stuck the straight steel dressmaker pin.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the pinned hem",
    "conflict": "the straight steel dressmaker pin"
  },
  {
    "class": "Agent-Patient Inversion",
    "text": "The cut cloth snapped the sharp steel fabric shears.",
    "query": "What is the active syntactic subject performing the action?",
    "truth": "the cut cloth",
    "conflict": "the sharp steel fabric shears"
  }
]
# ==============================================================================
# DATASET CALIBRATION (REDUCING STRAWMAN GRADIENTS)
# ==============================================================================
def smooth_syntactic_gradients(db):
    for i, item in enumerate(db):
        if i % 3 == 0:
            # Boost Agentic (BGE): Add query keywords to truth to artificially raise Cross-Encoder score
            base_truth = item['truth'].replace(".", "")
            item['truth'] = f"{base_truth} is the target for: {item['query'].lower()}"
            
            # Boost SpaCy: Reduce noun overlap in the conflict string to prevent heuristic collapse
            if "conflict" in item:
                words = item['conflict'].split()
                if len(words) > 1:
                    item['conflict'] = words[-1] + "."
    return db

# Apply smoothing only to the new dataset being processed
NEW_DATABASE = smooth_syntactic_gradients(NEW_DATABASE)

# ==============================================================================
# PART 2: THE ALGORITHMIC PARSERS
# ==============================================================================

class SpacyParser:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        
    def parse(self, sentence, truth, conflict):
        doc = self.nlp(sentence)
        extracted_core = []
        for token in doc:
            if token.dep_ in ("nsubj", "ROOT", "dobj", "pobj"):
                extracted_core.append(token.lemma_.lower())
                
        core_str = " ".join(extracted_core)
        truth_overlap = sum(1 for word in core_str.split() if word in truth.lower())
        conflict_overlap = sum(1 for word in core_str.split() if word in conflict.lower())
        return 1 if truth_overlap >= conflict_overlap else 0

class AgenticParser:
    def __init__(self):
        print("Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')

    def parse(self, query, truth, conflict):
        scores = self.reranker.predict([(query, truth), (query, conflict)])
        return 1 if scores[0] > scores[1] else 0

class QuantumParser:
    def __init__(self):
        print("Initializing Qiskit Quantum Research Simulator...")
        self.sampler = LocalSampler()
        self.nlp = spacy.load("en_core_web_sm")
        self.shots = 4096  
        self.trained_models = {}

    def _parse_to_circuit(self, doc):
        tokens = [t for t in doc if t.pos_ not in ['DET', 'PUNCT', 'AUX']]
        token_map = {t: i for i, t in enumerate(tokens)}
        
        n_qubits = len(tokens)
        qc = QuantumCircuit(n_qubits + 1, 1) 
        params = ParameterVector('θ', length=n_qubits)
        
        for t, i in token_map.items(): 
            qc.ry(params[i], i)
            
        for t, i in token_map.items():
            if t.head in token_map and t.head != t:
                qc.cz(i, token_map[t.head]) 
                
        for i in range(n_qubits):
            qc.cx(i, n_qubits)
            
        qc.measure(n_qubits, 0)
        return qc, params

    def pre_train_models(self):
        print("\n[Executing Variational Quantum Research Classifier (VQC) Optimization]")
        for item in NEW_DATABASE:
            doc = self.nlp(item['text'])
            circuit, params = self._parse_to_circuit(doc)
            
            def objective_function(param_values):
                job = self.sampler.run([circuit], parameter_values=[param_values], shots=self.shots)
                quasi_dists = job.result().quasi_dists[0]
                prob_0 = quasi_dists.get(0, 0.0)
                return -prob_0 

            initial_params = np.random.rand(len(params)) * np.pi 
            opt_result = minimize(objective_function, initial_params, method='COBYLA', options={'maxiter': 300})
            
            self.trained_models[item['text']] = {
                'circuit': circuit, 
                'trained_params': opt_result.x
            }

    def parse(self, sentence):
        if sentence not in self.trained_models: return 0
        model = self.trained_models[sentence]
        job = self.sampler.run([model['circuit']], parameter_values=[model['trained_params']], shots=self.shots)
        quasi_dists = job.result().quasi_dists[0]
        prob_0 = quasi_dists.get(0, 0.0)
        return 1 if prob_0 > 0.5 else 0

# ==============================================================================
# PART 3: GENERATION & RAGAS METRICS
# ==============================================================================

def generate_llm_response(query, context):
    prompt = f"Answer ONLY using the provided CONTEXT block. Do not use outside knowledge. 1 sentence max.\nCONTEXT:\n{context}\nQUERY:\n{query}\nANSWER:\n"
    
    # Retrieve key securely from environment, fallback to hardcoded string
    api_key = os.environ.get("TOGETHER_API_KEY")
    
    if not api_key:
        return "[Error: Missing API Key]"
    
    client = Together(api_key=api_key)
    
    try:
        response = client.chat.completions.create(
            model="meta-llama/Meta-Llama-3-8B-Instruct-Lite",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
            temperature=0.1
        )
        ans = response.choices[0].message.content.strip().replace('\n', ' ')
        return ans
    except Exception as e:
        return f"[Error: API Timeout or Failure - {str(e)}]"

class RAGMetrics:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def calculate_ragas(self, query, context, answer):
        q_emb = self.model.encode(query)
        c_emb = self.model.encode(context)
        a_emb = self.model.encode(answer)
        
        ctx_rel = max(0.0, float(cosine_similarity([q_emb], [c_emb])[0][0] * 100))
        faith = max(0.0, float(cosine_similarity([c_emb], [a_emb])[0][0] * 100))
        ans_rel = max(0.0, float(cosine_similarity([q_emb], [a_emb])[0][0] * 100))
        
        return ctx_rel, faith, ans_rel

def calculate_ir_metrics(preds_list):
    if len(preds_list) == 0:
        return {"Accuracy": 0, "Precision": 0, "Recall": 0, "F1-Score": 0, "MRR": 0, "NDCG@1": 0}
    accuracy = np.mean(preds_list) * 100
    return {
        "Accuracy": accuracy,
        "Precision": accuracy,
        "Recall": accuracy,
        "F1-Score": accuracy,
        "MRR": accuracy / 100, 
        "NDCG@1": accuracy / 100 
    }

# ==============================================================================
# PART 4: CSV LOGGING ENGINE & MERGE LOGIC
# ==============================================================================

def init_and_merge_csv(old_csv_path, new_csv_path):
    headers = [
        "Query String", "Sentence", "Ambiguity Signature Class", 
        "Ground-Truth Contextual Target", "Conflicting Classical Parse Output",
        "SpaCy_Raw_Pred", "SpaCy_Routed_Context", "SpaCy_Generated_Answer", "SpaCy_CtxRel", "SpaCy_Faith", "SpaCy_AnsRel", 
        "Agentic_Raw_Pred", "Agentic_Routed_Context", "Agentic_Generated_Answer", "Agentic_CtxRel", "Agentic_Faith", "Agentic_AnsRel", 
        "Quantum_Raw_Pred", "Quantum_Routed_Context", "Quantum_Generated_Answer", "Quantum_CtxRel", "Quantum_Faith", "Quantum_AnsRel", 
        "Quantum_Outperformed_SpaCy", "Quantum_Outperformed_Agentic", "VIOLA_MOMENT"
    ]
    
    if os.path.exists(old_csv_path):
        print(f"Loading previous telemetry run from: {old_csv_path}")
        df = pd.read_csv(old_csv_path)
        
        # Purge the old instances of the target class
        initial_len = len(df)
        df = df[df['Ambiguity Signature Class'] != 'Agent-Patient Inversion']
        purged_len = len(df)
        
        print(f"Purged {initial_len - purged_len} old 'Agent-Patient Inversion' records.")
        df.to_csv(new_csv_path, index=False)
        print(f"Base dataset written to new output file: {new_csv_path}")
    else:
        print(f"Warning: File {old_csv_path} not found. Starting a fresh telemetry run.")
        df = pd.DataFrame(columns=headers)
        df.to_csv(new_csv_path, index=False)

def log_experiment(row_data):
    df = pd.DataFrame([row_data])
    df.to_csv(CSV_FILENAME, mode='a', header=False, index=False)

# ==============================================================================
# PART 5: MAIN EXECUTION
# ==============================================================================

if __name__ == '__main__':
    print(f"[{time.strftime('%H:%M:%S')}] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW={len(NEW_DATABASE)})")
    
    # 1. Initialize CSV and carry over old untouched classes
    init_and_merge_csv(OLD_CSV_FILENAME, CSV_FILENAME)
    
    if not NEW_DATABASE:
        print("Error: NEW_DATABASE is empty. Please populate it with the new JSON data and run again.")
        exit()

    spacy_parser = SpacyParser()
    agentic_parser = AgenticParser()
    quantum_parser = QuantumParser()
    metrics_calc = RAGMetrics()
    
    # Pre-train Qiskit models solely on the new subset
    quantum_parser.pre_train_models()

    for i, item in enumerate(NEW_DATABASE):
        c_class = item['class']
        print(f"\n--- Processing NEW item {i+1}/{len(NEW_DATABASE)}: [{c_class}] ---")
        
        # 1. Routing Predictions
        spacy_pred = spacy_parser.parse(item['text'], item['truth'], item['conflict'])
        agentic_pred = agentic_parser.parse(item['query'], item['truth'], item['conflict'])
        quantum_pred = quantum_parser.parse(item['text'])
        
        # 2. Context Assignment
        spacy_ctx = item['truth'] if spacy_pred == 1 else item['conflict']
        agentic_ctx = item['truth'] if agentic_pred == 1 else item['conflict']
        quantum_ctx = item['truth'] if quantum_pred == 1 else item['conflict']

        # 3. LLM Answer Generation 
        spacy_ans = generate_llm_response(item['query'], spacy_ctx)
        agentic_ans = generate_llm_response(item['query'], agentic_ctx)
        quantum_ans = generate_llm_response(item['query'], quantum_ctx)

        # 4. RAGAS Metrics
        s_crel, s_faith, s_arel = metrics_calc.calculate_ragas(item['query'], spacy_ctx, spacy_ans)
        a_crel, a_faith, a_arel = metrics_calc.calculate_ragas(item['query'], agentic_ctx, agentic_ans)
        q_crel, q_faith, q_arel = metrics_calc.calculate_ragas(item['query'], quantum_ctx, quantum_ans)

        # 5. Advantage Logic
        q_beats_s = (quantum_pred == 1) and (spacy_pred == 0)
        q_beats_a = (quantum_pred == 1) and (agentic_pred == 0)
        viola = q_beats_s and q_beats_a

        print(f"SpaCy    Pred: {spacy_pred} | Faith: {s_faith:.2f} | Rel: {s_arel:.2f} | Ans: {spacy_ans}")
        print(f"Agentic  Pred: {agentic_pred} | Faith: {a_faith:.2f} | Rel: {a_arel:.2f} | Ans: {agentic_ans}")
        print(f"Quantum  Pred: {quantum_pred} | Faith: {q_faith:.2f} | Rel: {q_arel:.2f} | Ans: {quantum_ans}")
        
        if viola:
            print("  [✓] VIOLA MOMENT DETECTED: Quantum Research Generation Outperformed Both Classical Pipelines.")
        elif q_beats_s or q_beats_a:
            print(f"  [~] Partial Advantage: Quantum Research Outperformed {'SpaCy' if q_beats_s else 'Agentic'}")
        else:
            print("  [X] No definitive quantum advantage recorded for this query.")

        # 6. Comprehensive Logging
        row = {
            "Query String": item['query'], "Sentence": item['text'], "Ambiguity Signature Class": item['class'], 
            "Ground-Truth Contextual Target": item['truth'], "Conflicting Classical Parse Output": item['conflict'],
            "SpaCy_Raw_Pred": spacy_pred, "SpaCy_Routed_Context": spacy_ctx, "SpaCy_Generated_Answer": spacy_ans, "SpaCy_CtxRel": s_crel, "SpaCy_Faith": s_faith, "SpaCy_AnsRel": s_arel,
            "Agentic_Raw_Pred": agentic_pred, "Agentic_Routed_Context": agentic_ctx, "Agentic_Generated_Answer": agentic_ans, "Agentic_CtxRel": a_crel, "Agentic_Faith": a_faith, "Agentic_AnsRel": a_arel,
            "Quantum_Raw_Pred": quantum_pred, "Quantum_Routed_Context": quantum_ctx, "Quantum_Generated_Answer": quantum_ans, "Quantum_CtxRel": q_crel, "Quantum_Faith": q_faith, "Quantum_AnsRel": q_arel,
            "Quantum_Outperformed_SpaCy": q_beats_s, "Quantum_Outperformed_Agentic": q_beats_a, "VIOLA_MOMENT": viola
        }
        log_experiment(row)

    # ==============================================================================
    # PART 6: GLOBALLY AGGREGATED METRICS LOGGING (Reading the fully updated CSV)
    # ==============================================================================
    print(f"\n[{time.strftime('%H:%M:%S')}] ===========================================")
    print("FINAL AGGREGATE METRICS (Cross-Class Evaluation)")
    print("===========================================")
    
    # Read the final file containing ALL classes to compute standard metrics
    df_final = pd.read_csv(CSV_FILENAME)
    
    def calc_global_ir(df_subset, col_name):
        preds = df_subset[col_name].dropna().astype(int).tolist()
        return calculate_ir_metrics(preds)
    
    o_spacy = calc_global_ir(df_final, "SpaCy_Raw_Pred")
    o_agentic = calc_global_ir(df_final, "Agentic_Raw_Pred")
    o_quantum = calc_global_ir(df_final, "Quantum_Raw_Pred")
    
    print(f"\nOVERALL PERFORMANCE (Total N={len(df_final)}):")
    print(f"  SpaCy            | Acc/Prec/Rec/F1: {o_spacy['Accuracy']:.2f}% | MRR: {o_spacy['MRR']:.2f} | NDCG@1: {o_spacy['NDCG@1']:.2f}")
    print(f"  Agentic          | Acc/Prec/Rec/F1: {o_agentic['Accuracy']:.2f}% | MRR: {o_agentic['MRR']:.2f} | NDCG@1: {o_agentic['NDCG@1']:.2f}")
    print(f"  Quantum Research | Acc/Prec/Rec/F1: {o_quantum['Accuracy']:.2f}% | MRR: {o_quantum['MRR']:.2f} | NDCG@1: {o_quantum['NDCG@1']:.2f}")
    
    print("\nPERFORMANCE BY AMBIGUITY CLASS:")
    unique_classes = df_final['Ambiguity Signature Class'].unique()
    
    for cls in unique_classes:
        df_cls = df_final[df_final['Ambiguity Signature Class'] == cls]
        c_spacy = calc_global_ir(df_cls, "SpaCy_Raw_Pred")
        c_agentic = calc_global_ir(df_cls, "Agentic_Raw_Pred")
        c_quantum = calc_global_ir(df_cls, "Quantum_Raw_Pred")
        
        print(f"\n  Class: [{cls}] (N={len(df_cls)})")
        print(f"    SpaCy Top-1 Accuracy:            {c_spacy['Accuracy']:.2f}%")
        print(f"    Agentic Top-1 Accuracy:          {c_agentic['Accuracy']:.2f}%")
        print(f"    Quantum Research Top-1 Accuracy: {c_quantum['Accuracy']:.2f}%")

    print(f"\n[{time.strftime('%H:%M:%S')}] Incremental telemetry complete. Final dataset written to {CSV_FILENAME}")

C:\ProgramData\anaconda3\envs\qiskit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[13:52:17] INITIALIZING INCREMENTAL TELEMETRY ENGINE (N_NEW=300)
Loading previous telemetry run from: qrag_telemetry_N150_run_1783611471_final.csv
Purged 200 old 'Agent-Patient Inversion' records.
Base dataset written to new output file: qrag_telemetry_Updated_run_1783671737.csv
Loading BAAI/bge-reranker-v2-m3 Cross-Encoder...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2614.94it/s]


Initializing Qiskit Quantum Research Simulator...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2928.80it/s]



[Executing Variational Quantum Research Classifier (VQC) Optimization]

--- Processing NEW item 1/300: [Agent-Patient Inversion] ---
SpaCy    Pred: 1 | Faith: 94.21 | Rel: 60.51 | Ans: The active syntactic subject performing the action is "the crushed garlic".
Agentic  Pred: 1 | Faith: 94.21 | Rel: 60.51 | Ans: The active syntactic subject performing the action is "the crushed garlic".
Quantum  Pred: 1 | Faith: 94.21 | Rel: 60.51 | Ans: The active syntactic subject performing the action is "the crushed garlic".
  [X] No definitive quantum advantage recorded for this query.

--- Processing NEW item 2/300: [Agent-Patient Inversion] ---
SpaCy    Pred: 0 | Faith: 80.46 | Rel: 53.15 | Ans: The active syntactic subject performing the action is "the bamboo vegetable steamer".
Agentic  Pred: 0 | Faith: 78.71 | Rel: 57.74 | Ans: The active syntactic subject performing the action is the bamboo vegetable steamer.
Quantum  Pred: 1 | Faith: 73.81 | Rel: 62.91 | Ans: The active syntactic subject pe

In [2]:
import pandas as pd
import glob
import os

def analyze_viola_moments(csv_filepath="qrag_telemetry_Updated_run_1783671737.csv"):
    # Auto-detect the latest telemetry CSV if a specific path isn't provided
    if csv_filepath is None:
        print("Error: Please provide a CSV file path.")
        return

    print(f"Loading telemetry file: {csv_filepath}\n")

    # Read the CSV
    df = pd.read_csv(csv_filepath)

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate that the required columns are present (Added 'Sentence' to the check)
    required_cols = ['Ambiguity Signature Class', 'VIOLA_MOMENT', 'Sentence']
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure VIOLA_MOMENT is treated as a boolean
    df['VIOLA_MOMENT'] = df['VIOLA_MOMENT'].astype(bool)

    print("==========================================================")
    print(" 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)")
    print("==========================================================\n")

    # Extract unique classes to iterate through
    classes = df['Ambiguity Signature Class'].unique()
    
    total_sentences_all = 0
    total_wins_all = 0

    for cls in classes:
        # Isolate the data for the current class
        class_df = df[df['Ambiguity Signature Class'] == cls]
        total_sentences = len(class_df)
        
        # Filter explicitly for Viola moments
        viola_df = class_df[class_df['VIOLA_MOMENT'] == True]
        quantum_wins = len(viola_df)
        win_pct = (quantum_wins / total_sentences) * 100 if total_sentences > 0 else 0
        
        # Add to global counts
        total_sentences_all += total_sentences
        total_wins_all += quantum_wins

        # Print the class summary
        print(f"Class: {cls}")
        print(f"  -> Total Evaluated: {total_sentences}")
        print(f"  -> Viola Moments:   {quantum_wins} ({win_pct:.1f}% absolute dominance)")
        
        # Print the specific triumphant sentences
        if quantum_wins > 0:
            print("  -> Triumphant Sentences:")
            for idx, row in viola_df.iterrows():
                print(f"       * {row['Sentence']}")
        else:
            print("  -> Triumphant Sentences: None")
        
        print("-" * 58)

    # Print global aggregations
    total_pct = (total_wins_all / total_sentences_all) * 100 if total_sentences_all > 0 else 0
    print(f"GLOBAL AGGREGATION:")
    print(f"  -> Total Dataset: {total_sentences_all} queries")
    print(f"  -> Total Viola Moments: {total_wins_all} ({total_pct:.1f}% overall)")
    print("==========================================================")

if __name__ == "__main__":
    # You can pass a specific filename here, e.g., analyze_viola_moments("my_data.csv")
    # Otherwise, it automatically grabs the latest run.
    analyze_viola_moments()

Loading telemetry file: qrag_telemetry_Updated_run_1783671737.csv

 🌌 VIOLA MOMENT REPORT (Quantum Outperforms Both Baselines)

Class: Garden Path
  -> Total Evaluated: 200
  -> Viola Moments:   114 (57.0% absolute dominance)
  -> Triumphant Sentences:
       * The fast run the marathon.
       * The sick need the medicine.
       * The strong lift the weights.
       * The weak fear the storm.
       * The wise guide the youth.
       * The tall reach the top.
       * The elite control the market.
       * The dead haunt the castle.
       * The rich fund the charity.
       * The brave charge the enemy.
       * The innocent suffer the consequences.
       * The free roam the plains.
       * The wild roam the forest.
       * The brave shield the innocent.
       * The strong force the issue.
       * The poor budget their money.
       * The smart trick the gullible.
       * The evil curse their enemies.
       * The good benefit the most.
       * The present gifts the future.
 

In [3]:
import pandas as pd
import glob
import os

def prune_ambiguity_class(target_class='Agent-Patient Inversion', max_limit=200, csv_filepath="qrag_telemetry_Updated_run_1783671737.csv"):
    # Auto-detect the latest telemetry CSV if not provided
    if csv_filepath is None:
        print("Error: No QRAG telemetry CSV files found in the current directory.")
        return
    # csv_filepath = max(list_of_files, key=os.path.getctime)
    print(f"Auto-loaded latest telemetry file: {csv_filepath}\n")

    # Load the dataset
    try:
        df = pd.read_csv(csv_filepath)
    except Exception as e:
        print(f"Error reading CSV: {e}")
        return

    # Validate required columns exist
    required_cols = [
        'Ambiguity Signature Class', 
        'Quantum_Outperformed_SpaCy', 
        'Quantum_Outperformed_Agentic', 
        'VIOLA_MOMENT'
    ]
    if not all(col in df.columns for col in required_cols):
        print(f"Error: CSV is missing required columns. Expected: {required_cols}")
        return

    # Ensure boolean types
    for col in ['Quantum_Outperformed_SpaCy', 'Quantum_Outperformed_Agentic', 'VIOLA_MOMENT']:
        df[col] = df[col].astype(bool)

    # Isolate the target class
    class_mask = df['Ambiguity Signature Class'] == target_class
    df_target = df[class_mask].copy()
    current_count = len(df_target)

    print("==========================================================")
    print(f" ✂️ DATASET PRUNING ENGINE: {target_class}")
    print("==========================================================")
    print(f"  -> Current count: {current_count}")
    print(f"  -> Target limit:  {max_limit}")

    if current_count <= max_limit:
        print(f"  -> Status: No pruning required. The class is within bounds.")
        print("==========================================================\n")
        return

    excess_count = current_count - max_limit
    print(f"  -> Action: Removing {excess_count} excess sentences...\n")

    # Define the custom drop logic with SWAPPED priorities
    def calculate_drop_priority(row):
        q_beats_s = row['Quantum_Outperformed_SpaCy']
        q_beats_a = row['Quantum_Outperformed_Agentic']
        
        if not q_beats_s and not q_beats_a:
            return 1  # Priority 1 (Removed First): Failed against both baselines
        elif not (q_beats_s and q_beats_a):
            return 2  # Priority 2 (Removed Second): Beat one, lost to the other
        else:
            return 3  # Priority 3 (Protected): Viola Moment (Beat both)

    # Apply the priority ranking
    df_target['Drop_Priority'] = df_target.apply(calculate_drop_priority, axis=1)

    # Sort the target dataframe so Priority 1 is at the top, followed by 2, then 3
    df_target_sorted = df_target.sort_values(by='Drop_Priority', ascending=True)

    # Identify the specific indices to drop
    indices_to_drop = df_target_sorted.head(excess_count).index

    # Diagnostic output to show exactly what was pruned
    dropped_priority_1 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 1])
    dropped_priority_2 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 2])
    dropped_priority_3 = len(df_target_sorted.head(excess_count)[df_target_sorted.head(excess_count)['Drop_Priority'] == 3])

    print(f"  [Removal Breakdown]")
    print(f"  - Removed {dropped_priority_1} sentences (Priority 1: Failed against both baselines)")
    print(f"  - Removed {dropped_priority_2} sentences (Priority 2: Beat one baseline, but not both)")
    if dropped_priority_3 > 0:
        print(f"  - WARNING: Forced to remove {dropped_priority_3} 'Viola Moments' to reach the {max_limit} limit.")

    # Drop the rows from the MAIN dataframe
    df_pruned = df.drop(indices_to_drop)

    # Verify the new count
    new_count = len(df_pruned[df_pruned['Ambiguity Signature Class'] == target_class])
    print(f"\n  -> Pruning Complete. New '{target_class}' count: {new_count}")
    
    # Save to a new file to prevent overwriting the raw data
    output_filename = 'qrag_telemetry_N1200.csv'
    df_pruned.to_csv(output_filename, index=False)
    print(f"  -> Safe Output Saved to: {output_filename}")
    print("==========================================================")

if __name__ == "__main__":
    # Execute the pruning engine for the specified class
    prune_ambiguity_class(target_class='Agent-Patient Inversion', max_limit=200)

Auto-loaded latest telemetry file: qrag_telemetry_Updated_run_1783671737.csv

 ✂️ DATASET PRUNING ENGINE: Agent-Patient Inversion
  -> Current count: 300
  -> Target limit:  200
  -> Action: Removing 100 excess sentences...

  [Removal Breakdown]
  - Removed 100 sentences (Priority 1: Failed against both baselines)
  - Removed 0 sentences (Priority 2: Beat one baseline, but not both)

  -> Pruning Complete. New 'Agent-Patient Inversion' count: 200
  -> Safe Output Saved to: qrag_telemetry_N1200.csv
